In [0]:
# Bronze layer creation (orders_raw.csv)
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from datetime import datetime
import traceback
from typing import Optional

CONFIG = {
    "source_file": "/Workspace/Users/{USEER_EMAIL}/orders_raw.csv",
    "catalog": "medallion_example",
    "schema": "a_bronze",
    "table": "orders_raw",
    "table_location": None 
}

TARGET_TABLE = f"{CONFIG['catalog']}.{CONFIG['schema']}.{CONFIG['table']}"

def log(message: str) -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {message}")

def create_catalog_and_schema(catalog_name: str, schema_name: str) -> None:
    log(f"Initializing infrastructure: Catalog '{catalog_name}', Schema '{schema_name}'")
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
    log(f"Infrastructure ready: {catalog_name}.{schema_name}")

def read_source_file(path: str) -> DataFrame:
    log(f"Reading source file: {path}")
    return (
        spark.read
        .option("header", "true")
        .option("mode", "PERMISSIVE") 
        .option("inferSchema", "false")
        .csv(path)
    )

def add_metadata_columns(df: DataFrame, column_name: str) -> DataFrame:
    log(f"Adding metadata column: {column_name}")
    return df.withColumn(column_name, F.current_timestamp())

def write_delta_table(
    df: DataFrame,
    table_name: str,
    table_location: Optional[str] = None,
    mode: str = "overwrite"
) -> None:
    log(f"Writing table: {table_name}")
    writer = df.write.format("delta").mode(mode)

    if table_location:
        log(f"Using custom external location: {table_location}")
        writer.option("path", table_location).saveAsTable(table_name)
    else:
        log("Using managed UC table location")
        writer.saveAsTable(table_name)
    log("Write completed")

def validate_table(table_name: str) -> None:
    df = spark.table(table_name)
    log(f"Validation table: {table_name}")
    log(f"Row count: {df.count()}")
    df.printSchema()
    display(df.limit(50))

def run_bronze_load() -> None:
    start_time = datetime.now()
    try:
        log("=" * 80)
        log("BRONZE LOAD STARTED")
        
        create_catalog_and_schema(CONFIG["catalog"], CONFIG["schema"])

        raw_df = read_source_file(CONFIG["source_file"])
        bronze_df = add_metadata_columns(raw_df, "_ingestion_ts")

        write_delta_table(bronze_df, TARGET_TABLE)
        validate_table(TARGET_TABLE)

        duration = (datetime.now() - start_time).total_seconds()
        log("=" * 80)
        log("BRONZE LOAD SUCCESS")
        log(f"Target table: {TARGET_TABLE}")
        log(f"Duration: {duration:.2f} seconds")
        log("=" * 80)

    except Exception as ex:
        log("=" * 80)
        log("BRONZE LOAD FAILED")
        log("=" * 80)
        traceback.print_exc()
        raise

run_bronze_load()

[2026-10-05 08:09:45] ================================================================================
[2026-10-05 08:09:45] BRONZE LOAD STARTED
[2026-10-05 08:09:45] Initializing infrastructure: Catalog 'medallion_example', Schema 'a_bronze'
[2026-10-05 08:09:46] Infrastructure ready: medallion_example.a_bronze
[2026-10-05 08:09:46] Reading source file: /Workspace/Users/shahniuke@gmail.com/orders_raw.csv
[2026-10-05 08:09:46] Adding metadata column: _ingestion_ts
[2026-10-05 08:09:46] Writing table: medallion_example.a_bronze.orders_raw
[2026-10-05 08:09:46] Using managed UC table location
[2026-10-05 08:09:50] Write completed
[2026-10-05 08:09:50] Validation table: medallion_example.a_bronze.orders_raw
[2026-10-05 08:09:50] Row count: 50
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullabl

order_id,customer_id,order_date,region,product_category,product_name,quantity,unit_price,total_amount,_ingestion_ts
3001,CUST2001,2025-01-01,North,Software,Project Management Pro,2,-25.99,51.98,2026-10-05T08:09:48.637Z
3002,CUST2002,2025-01-02,South,Cloud Services,Object Storage,1,14.99,14.99,2026-10-05T08:09:48.637Z
3003,CUST2003,2025-01-03,East,Analytics,BI Dashboard License,3,-29.50,88.50,2026-10-05T08:09:48.637Z
3004,CUST2004,2025-01-04,West,Security,Endpoint Protection,1,119.99,119.99,2026-10-05T08:09:48.637Z
3005,CUST2005,2025-01-05,North,Cloud Services,Managed Database,2,35.75,71.50,2026-10-05T08:09:48.637Z
3006,CUST2006,2025-01-06,South,Analytics,Data Warehouse License,1,-22.99,22.99,2026-10-05T08:09:48.637Z
3007,CUST2007,2025-01-07,East,Security,Identity Manager,4,18.50,74.00,2026-10-05T08:09:48.637Z
3008,CUST2008,2025-01-08,West,Cloud Services,Virtual Machine,5,9.99,49.95,2026-10-05T08:09:48.637Z
3009,CUST2009,2025-01-09,North,Developer Tools,Git Enterprise,1,149.99,149.99,2026-10-05T08:09:48.637Z
3010,CUST2010,2025-01-10,South,Analytics,Data Catalog,2,29.99,59.98,2026-10-05T08:09:48.637Z


[2026-10-05 08:09:51] ================================================================================
[2026-10-05 08:09:51] BRONZE LOAD SUCCESS
[2026-10-05 08:09:51] Target table: medallion_example.a_bronze.orders_raw
[2026-10-05 08:09:51] Duration: 6.07 seconds
[2026-10-05 08:09:51] ================================================================================


# Bronze Layer - Raw Data Ingestion

## Objective
The Bronze layer acts as an append-only landing zone that captures the raw source data in its original form. Processing is strictly limited to format conversion to ensure maximum throughput and full data auditability.

---

## Ingestion Strategy
* **Format:** Raw CSV ingested and written directly to a native Delta table.
* **Schema:** Original column structures and data types are preserved without modification.
* **Transformations:** Zero business logic, data cleansing, or filtering applied.
* **Data Quality:** Validation checks are deferred entirely to the Silver layer to avoid ingestion bottlenecks.

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from datetime import datetime
import traceback
from typing import Optional

CONFIG = {
    "catalog": "medallion_example",
    "bronze_schema": "a_bronze",
    "bronze_table": "orders_raw",
    "silver_schema": "a_silver",
    "silver_table": "orders_clean",
    "quarantine_schema": "a_quarantine",
    "quarantine_table": "orders_invalid",
    "silver_location": "",
    "quarantine_location": ""
}

BRONZE_TABLE = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{CONFIG['bronze_table']}"
SILVER_TABLE = f"{CONFIG['catalog']}.{CONFIG['silver_schema']}.{CONFIG['silver_table']}"
QUARANTINE_TABLE = f"{CONFIG['catalog']}.{CONFIG['quarantine_schema']}.{CONFIG['quarantine_table']}"

spark.conf.set("spark.sql.ansi.enabled", "false")

def log(message: str) -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {message}")

def create_schema(catalog_name: str, schema_name: str) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
    log(f"Schema ready: {catalog_name}.{schema_name}")

def read_bronze_table(table_name: str) -> DataFrame:
    log(f"Reading bronze table: {table_name}")
    return spark.table(table_name)

def standardize_columns(df: DataFrame) -> DataFrame:
    log("Standardizing source columns")
    return (
        df
        .withColumn("order_id_int", F.col("order_id").cast("bigint"))
        .withColumn(
            "order_date_clean",
            F.coalesce(
                F.to_date("order_date", "yyyy-MM-dd"),
                F.to_date("order_date", "yyyy/MM/dd"),
                F.to_date("order_date", "MM/dd/yyyy"),
                F.to_date("order_date", "dd-MM-yyyy")
            )
        )
        .withColumn("region_clean", F.initcap(F.lower(F.col("region"))))
        .withColumn("quantity_int", F.col("quantity").cast("int"))
        .withColumn(
            "unit_price_num",
            F.when(F.col("unit_price") == "N/A", None).otherwise(F.col("unit_price").cast("double"))
        )
        .withColumn("total_amount_num", F.col("total_amount").cast("double"))
    )

def apply_data_quality_rules(df: DataFrame) -> DataFrame:
    log("Applying DQ validation rules")
    return (
        df
        .withColumn(
            "dq_reason",
            F.when(F.col("order_id_int").isNull(), "INVALID_ORDER_ID")
            .when(F.col("customer_id").isNull(), "MISSING_CUSTOMER_ID")
            .when(F.col("order_date_clean").isNull(), "INVALID_ORDER_DATE")
            .when(F.col("region_clean").isNull(), "MISSING_REGION")
            .when(F.col("product_name").isNull(), "MISSING_PRODUCT")
            .when(F.col("quantity_int").isNull(), "MISSING_QUANTITY")
            .when(F.col("quantity_int") <= 0, "INVALID_QUANTITY")
            .when(F.col("unit_price_num").isNull(), "MISSING_UNIT_PRICE")
            .when(F.col("unit_price_num") <= 0, "INVALID_UNIT_PRICE")
            .otherwise(None)
        )
    )

def create_quarantine_dataframe(df: DataFrame) -> DataFrame:
    return (
        df.filter(F.col("dq_reason").isNotNull())
        .withColumn("_quarantine_ts", F.current_timestamp())
    )

def create_silver_dataframe(df: DataFrame) -> DataFrame:
    silver_df = df.filter(F.col("dq_reason").isNull())
    silver_df = (
        silver_df
        .withColumn(
            "total_amount_final",
            F.coalesce(
                F.col("total_amount_num"),
                F.round(F.col("quantity_int") * F.col("unit_price_num"), 2)
            )
        )
    )

    silver_df = silver_df.dropDuplicates(["order_id_int"])

    return silver_df.select(
        F.col("order_id_int").alias("order_id"),
        "customer_id",
        F.col("order_date_clean").alias("order_date"),
        F.col("region_clean").alias("region"),
        "product_category",
        "product_name",
        F.col("quantity_int").alias("quantity"),
        F.col("unit_price_num").alias("unit_price"),
        F.col("total_amount_final").alias("total_amount")
    )

def write_delta_table(df: DataFrame, table_name: str, table_location: Optional[str] = None, mode: str = "overwrite") -> None:
    log(f"Writing table: {table_name}")
    writer = df.write.format("delta").mode(mode)

    if table_location:
        log(f"Using custom location: {table_location}")
        writer.option("path", table_location).saveAsTable(table_name)
    else:
        log("Using managed table location")
        writer.saveAsTable(table_name)
    log("Write completed")

def validate_table(table_name: str) -> None:
    df = spark.table(table_name)
    log(f"Validation table: {table_name}")
    log(f"Row count: {df.count()}")
    df.printSchema()
    display(df.limit(20))

def display_dq_summary(quarantine_df: DataFrame) -> None:
    display(quarantine_df.groupBy("dq_reason").count().orderBy(F.desc("count")))

def run_silver_load() -> None:
    start_time = datetime.now()
    try:
        log("=" * 80)
        log("SILVER LOAD STARTED")

        create_schema(CONFIG["catalog"], CONFIG["silver_schema"])
        create_schema(CONFIG["catalog"], CONFIG["quarantine_schema"])

        bronze_df = read_bronze_table(BRONZE_TABLE)
        standardized_df = standardize_columns(bronze_df)
        dq_df = apply_data_quality_rules(standardized_df)
        
        quarantine_df = create_quarantine_dataframe(dq_df)
        silver_df = create_silver_dataframe(dq_df)

        write_delta_table(quarantine_df, QUARANTINE_TABLE)
        write_delta_table(silver_df, SILVER_TABLE)

        validate_table(SILVER_TABLE)
        display_dq_summary(quarantine_df)

        duration = (datetime.now() - start_time).total_seconds()

        log("=" * 80)
        log("SILVER LOAD SUCCESS")
        log(f"Silver table: {SILVER_TABLE}")
        log(f"Quarantine table: {QUARANTINE_TABLE}")
        log(f"Duration: {duration:.2f}s")
        log("=" * 80)

    except Exception as ex:
        log("=" * 80)
        log("SILVER LOAD FAILED")
        log("=" * 80)
        print(str(ex))
        traceback.print_exc()
        raise

run_silver_load()

[2026-10-05 08:09:52] ================================================================================
[2026-10-05 08:09:52] SILVER LOAD STARTED
[2026-10-05 08:09:52] Schema ready: medallion_example.a_silver
[2026-10-05 08:09:53] Schema ready: medallion_example.a_quarantine
[2026-10-05 08:09:53] Reading bronze table: medallion_example.a_bronze.orders_raw
[2026-10-05 08:09:53] Standardizing source columns
[2026-10-05 08:09:53] Applying DQ validation rules
[2026-10-05 08:09:53] Writing table: medallion_example.a_quarantine.orders_invalid
[2026-10-05 08:09:53] Using managed table location
[2026-10-05 08:09:55] Write completed
[2026-10-05 08:09:55] Writing table: medallion_example.a_silver.orders_clean
[2026-10-05 08:09:55] Using managed table location
[2026-10-05 08:09:58] Write completed
[2026-10-05 08:09:58] Validation table: medallion_example.a_silver.orders_clean
[2026-10-05 08:09:58] Row count: 32
root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 

order_id,customer_id,order_date,region,product_category,product_name,quantity,unit_price,total_amount
3005,CUST2005,2025-01-05,North,Cloud Services,Managed Database,2,35.75,71.5
3009,CUST2009,2025-01-09,North,Developer Tools,Git Enterprise,1,149.99,149.99
3011,CUST2011,2025-01-11,East,Security,Vulnerability Scanner,2,39.99,79.98
3015,CUST2015,2025-01-15,East,Analytics,Reporting Engine,1,79.99,79.99
3013,CUST2013,2025-01-13,North,Software,CRM Platform,2,79.99,159.98
3014,CUST2014,2025-01-14,South,Developer Tools,CI/CD Pipeline,3,12.99,38.97
3016,CUST2016,2025-01-16,West,Cloud Services,Managed Spark Cluster,2,45.0,90.0
3024,CUST2024,2025-01-24,West,Analytics,Semantic Layer,3,45.0,135.0
3026,CUST2026,2025-01-26,South,Cloud Services,Data Lake Storage,2,80.0,160.0
3004,CUST2004,2025-01-04,West,Security,Endpoint Protection,1,119.99,119.99


dq_reason,count
INVALID_UNIT_PRICE,3
MISSING_QUANTITY,2
INVALID_QUANTITY,2
INVALID_ORDER_DATE,2
MISSING_CUSTOMER_ID,2
MISSING_PRODUCT,2
MISSING_UNIT_PRICE,2
MISSING_REGION,2
INVALID_ORDER_ID,1


[2026-10-05 08:10:00] ================================================================================
[2026-10-05 08:10:00] SILVER LOAD SUCCESS
[2026-10-05 08:10:00] Silver table: medallion_example.a_silver.orders_clean
[2026-10-05 08:10:00] Quarantine table: medallion_example.a_quarantine.orders_invalid
[2026-10-05 08:10:00] Duration: 8.02s
[2026-10-05 08:10:00] ================================================================================


# Silver Layer - Data Cleansing & Standardization

## Objective
Transform raw Bronze datasets into standardized, validated, and deduplicated tables ready for downstream analytics. All records failing data quality (DQ) checks are routed to an isolated Quarantine table without stopping the pipeline.

---

## Data Quality (DQ) Rules & Handling Matrix

| Target Column | Identified Issue | Transformation & Handling Strategy | Reason Code |
| :--- | :--- | :--- | :--- |
| `order_id` | Non-numeric / invalid IDs | Cast to `BIGINT`. Failures go to Quarantine. | `INVALID_ORDER_ID` |
| `order_date` | Mixed formats (`yyyy-MM-dd`, `yyyy/MM/dd`, `MM/dd/yyyy`, `dd-MM-yyyy`) | Parse and convert to standard Spark `DATE`. Unparseable go to Quarantine. | `INVALID_ORDER_DATE` |
| `region` | Inconsistent casing (`north`, `NORTH`), missing values | Apply `INITCAP()` formatting. Missing values go to Quarantine. | `MISSING_REGION` |
| `quantity` | Non-numeric, missing, or $\le 0$ values | Cast to `INT`. Missing, non-numeric, or values $\le 0$ go to Quarantine. | `MISSING_QUANTITY`<br>`INVALID_QUANTITY` |
| `unit_price` | `N/A` strings and non-numeric values | Replace `N/A` with `NULL`, then cast to `DOUBLE`. Failures go to Quarantine. | `INVALID_UNIT_PRICE` |
| `customer_id`<br>`product` | Missing critical attributes | Enforce `NOT NULL`. Missing rows go to Quarantine. | `MISSING_CUSTOMER_ID`<br>`MISSING_PRODUCT` |
| `total_amount`| Missing values | Keep original if exists. If missing, recalculate: `quantity * unit_price`. | *N/A (In-line fix)* |

### Deduplication Policy
* **Rule:** Deduplicate records using `order_id` as the primary business key, keeping the first valid record.
* *Note: For a full production environment, duplicate business keys should be routed to Quarantine for deep investigation instead of being silently dropped.*

---

## Production Enhancements (Roadmap)
To transition this pipeline from MVP to enterprise-grade, the following metadata-driven rules should be added:
1. **String Sanitization:** Automatic trimming of whitespaces and standardizing placeholder values (`null`, `NA`, `Unknown`) to true SQL `NULL`.
2. **Referential Integrity:** Validation of string fields against dynamic master data (e.g., allowed region codes or product taxonomies).
3. **Financial Hard Checks:** Cross-validation of metric logic (e.g., triggering a alert if `total_amount != quantity * unit_price`).

---

## Quarantine Layer Design
Quarantined records are safely preserved in a separate Delta table with additional execution metadata for debugging and reprocessing:
* `dq_reason`: Specific error tag or failed validation rule.
* `_quarantine_ts`: Verification timestamp.

---

## Final Silver Schema State
The output table guarantees: typed schema, standardized dates/casing, enforced mandatory business fields, recalculated financial totals, and zero duplicates.


In [0]:
# Gold layer creation (gold.sales_summary)
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from datetime import datetime
import traceback
from typing import Optional

CONFIG = {
    "catalog": "medallion_example",
    "silver_schema": "a_silver",
    "silver_table": "orders_clean",
    "gold_schema": "a_gold",
    "gold_table": "sales_summary",
    "gold_location": None
}

SILVER_TABLE = f"{CONFIG['catalog']}.{CONFIG['silver_schema']}.{CONFIG['silver_table']}"
GOLD_TABLE = f"{CONFIG['catalog']}.{CONFIG['gold_schema']}.{CONFIG['gold_table']}"

def log(message: str) -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {message}")

def create_schema(catalog_name: str, schema_name: str) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
    log(f"Schema ready: {catalog_name}.{schema_name}")

def read_silver_table(table_name: str) -> DataFrame:
    log(f"Reading silver table: {table_name}")
    return spark.table(table_name)

def create_gold_dataframe(df: DataFrame) -> DataFrame:
    log("Creating business reporting dataset")
    return (
        df
        .withColumn("order_month", F.date_format("order_date", "yyyy-MM"))
        .groupBy("order_month", "region")
        .agg(
            F.count("order_id").alias("orders_cnt"),
            F.countDistinct("customer_id").alias("customers_cnt"),
            F.sum("quantity").alias("total_quantity"),
            F.round(F.sum("total_amount"), 2).alias("total_sales"),
            F.round(F.avg("total_amount"), 2).alias("avg_order_value")
        )
    )

def write_delta_table(
    df: DataFrame, 
    table_name: str, 
    table_location: Optional[str] = None, 
    mode: str = "overwrite"
) -> None:
    log(f"Writing table: {table_name}")
    writer = df.write.format("delta").mode(mode)

    if table_location:
        writer.option("path", table_location).saveAsTable(table_name)
    else:
        writer.saveAsTable(table_name)
    log("Write completed")

def validate_table(table_name: str) -> None:
    df = spark.table(table_name)
    log(f"Validation table: {table_name}")
    log(f"Row count: {df.count()}")
    df.printSchema()
    display(df.limit(20))

def run_gold_load() -> None:
    start_time = datetime.now()
    try:
        log("=" * 80)
        log("GOLD LOAD STARTED")

        create_schema(CONFIG["catalog"], CONFIG["gold_schema"])

        silver_df = read_silver_table(SILVER_TABLE)
        gold_df = create_gold_dataframe(silver_df)

        write_delta_table(gold_df, GOLD_TABLE)
        validate_table(GOLD_TABLE)

        duration = (datetime.now() - start_time).total_seconds()
        log("=" * 80)
        log("GOLD LOAD SUCCESS")
        log(f"Gold table: {GOLD_TABLE}")
        log(f"Duration: {duration:.2f}s")
        log("=" * 80)

    except Exception as ex:
        log("=" * 80)
        log("GOLD LOAD FAILED")
        log("=" * 80)
        traceback.print_exc()
        raise

run_gold_load()

[2026-10-05 08:10:00] ================================================================================
[2026-10-05 08:10:00] GOLD LOAD STARTED
[2026-10-05 08:10:01] Schema ready: medallion_example.a_gold
[2026-10-05 08:10:01] Reading silver table: medallion_example.a_silver.orders_clean
[2026-10-05 08:10:01] Creating business reporting dataset
[2026-10-05 08:10:01] Writing table: medallion_example.a_gold.sales_summary
[2026-10-05 08:10:03] Write completed
[2026-10-05 08:10:03] Validation table: medallion_example.a_gold.sales_summary
[2026-10-05 08:10:04] Row count: 8
root
 |-- order_month: string (nullable = true)
 |-- region: string (nullable = true)
 |-- orders_cnt: long (nullable = true)
 |-- customers_cnt: long (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- avg_order_value: double (nullable = true)



order_month,region,orders_cnt,customers_cnt,total_quantity,total_sales,avg_order_value
2025-02,West,1,1,2,104.0,104.0
2025-02,East,1,1,2,88.0,88.0
2025-01,East,7,7,18,516.97,73.85
2025-01,West,7,7,16,705.9,100.84
2025-01,South,7,7,13,521.92,74.56
2025-02,North,1,1,4,112.0,112.0
2025-02,South,1,1,3,63.0,63.0
2025-01,North,7,7,14,858.44,122.63


[2026-10-05 08:10:05] ================================================================================
[2026-10-05 08:10:05] GOLD LOAD SUCCESS
[2026-10-05 08:10:05] Gold table: medallion_example.a_gold.sales_summary
[2026-10-05 08:10:05] Duration: 4.72s
[2026-10-05 08:10:05] ================================================================================


# Gold Layer - Business Reporting Dataset

## Objective
The Gold layer provides a curated, business-ready dataset optimized for performance, BI reporting (Power BI, Tableau), and downstream analytics.

---

## Aggregation Grain & Metrics

Data is pre-aggregated by **Order Month** and **Region**.

| Metric | Definition |
| :--- | :--- |
| `orders_cnt` | Unique order volume |
| `customers_cnt` | Unique customer count |
| `total_quantity` | Total units sold |
| `total_sales` | Total gross revenue |
| `avg_order_value` | Average order financial value |

---

## Business Value
* **Performance:** Eliminates the need for expensive, repetitive aggregations in BI tools.
* **Consistency:** Ensures a single source of truth for key financial and operational metrics.

# Governance & Access Control

## Access Matrix (Unity Catalog)

| Role | Bronze | Silver | Quarantine | Gold |
| :--- | :--- | :--- | :--- | :--- |
| **business_analyst** | No Access | No Access | No Access | Read |
| **data_engineer** | Read/Write/Create | Read/Write/Create | Read/Write/Create | Read/Write/Create |
| **compliance_auditor**| Read | Read | Read | Read |

*Note: Quarantine data is isolated in a separate `a_quarantine` schema to eliminate accidental access.*

---

## RBAC Setup

### 1. Business Analyst
```sql
GRANT USE SCHEMA, SELECT ON SCHEMA a_gold TO `business_analyst`;
GRANT SELECT ON FUTURE TABLES IN SCHEMA a_gold TO `business_analyst`;
```

### 2. Data Engineer
```sql
-- Bulk schema access
GRANT USE SCHEMA, CREATE TABLE ON SCHEMA a_bronze, a_silver, a_quarantine, a_gold TO `data_engineer`;

-- Data manipulation
GRANT SELECT, MODIFY ON SCHEMA a_bronze, a_silver, a_quarantine, a_gold TO `data_engineer`;
GRANT SELECT, MODIFY ON FUTURE TABLES IN SCHEMA a_bronze, a_silver, a_quarantine, a_gold TO `data_engineer`;
```

### 3. Compliance Auditor
```sql
GRANT USE SCHEMA ON SCHEMA a_bronze, a_silver, a_quarantine, a_gold TO `compliance_auditor`;

GRANT SELECT ON SCHEMA a_bronze, a_silver, a_quarantine, a_gold TO `compliance_auditor`;
GRANT SELECT ON FUTURE TABLES IN SCHEMA a_bronze, a_silver, a_quarantine, a_gold TO `compliance_auditor`;
```

### Explicit Revoke (Guardrail)
```sql
-- Ensure analysts have zero visibility into quarantine
REVOKE ALL PRIVILEGES ON SCHEMA a_quarantine FROM `business_analyst`;
```

---

## ABAC Example (Row-Level Security)

Fine-grained row filtering applied on top of RBAC based on the user's Entra ID region attribute.

### 1. Tag the security dimension
```sql
ALTER TABLE a_gold.sales_summary ALTER COLUMN region SET TAGS ('security_dimension' = 'region');
```

### 2. Create and apply row filter
```sql
CREATE FUNCTION a_gold.region_filter(region STRING)
RETURN current_user_attribute('region') = region;

ALTER TABLE a_gold.sales_summary SET ROW FILTER a_gold.region_filter ON (region);
```

### 3. Expected behavior
If `current_user_attribute('region')` returns `'East'`, the user only sees matching records:
```text
2026-05 | East | 8355.54
2026-06 | East | 5871.44
```
All other regions (North, South, West) are automatically filtered out at the engine level.


# Production Delivery

## Reporting View & Access Control

To protect the underlying Gold table and provide an abstraction layer for BI tools, analysts are granted access exclusively through a dedicated reporting view.

```sql
-- 1. Create secure presentation layer
CREATE OR REPLACE VIEW a_gold.v_sales_dashboard AS
SELECT
    order_month,
    region,
    orders_cnt,
    customers_cnt,
    total_quantity,
    total_sales,
    avg_order_value
FROM a_gold.sales_summary;

-- 2. Provision data access
GRANT SELECT ON VIEW a_gold.v_sales_dashboard TO `business_analyst`;
```

---

## Business Query Example

**Scenario:** Analyze regional sales performance, order volumes, and average order values to identify top-performing markets.

```sql
SELECT
    region,
    SUM(total_sales) AS total_sales,
    SUM(orders_cnt) AS total_orders,
    ROUND(AVG(avg_order_value), 2) AS avg_order_value
FROM a_gold.v_sales_dashboard
GROUP BY region
ORDER BY total_sales DESC;
```



# Automation

The pipeline is orchestrated using a Databricks Workflow with separate Bronze, Silver, Gold tasks. Silver processing includes data standardization, data quality validation and automatic routing of invalid records to a quarantine table.

Common configuration would be passed through Databricks Widgets and Job Parameters (catalog, environment, schemas, storage locations, processing date), ensuring a metadata-driven design and enabling the same notebooks to run across DEV, TEST and PROD without modifications.

To reduce code duplication and improve maintainability, common functionality such as logging, configuration management, data quality checks, Delta write operations, error handling and validation logic would be implemented as reusable Python modules or shared libraries. This allows the same components to be reused consistently across Bronze, Silver and Gold layers and simplifies future enhancements.

Deployment would be automated using Databricks Asset Bundles and CI/CD pipelines. The workflow would include retries, audit logging, data quality checks, monitoring dashboards, and automatic email/Teams notifications for failures, SLA breaches or unusual data quality metrics.